In [74]:
#Importing dependencies and modules

import pandas as pd
import seaborn as sns
import numpy as np
import tensorflow.keras as keras
import matplotlib.pyplot as plt

In [75]:
#Opening up and merging both train and test datasets

test_df = pd.read_csv("/users/imbahndu/Desktop/Columbia DBM/PFED5/test_cnn.csv")
train_df = pd.read_csv("/users/imbahndu/Desktop/Columbia DBM/PFED5/train_cnn.csv")

In [76]:
#Concatenate the dfs
df = pd.concat([train_df, test_df], axis = 0)

In [77]:
df

,UPDRS_score,main_frame
0,4,frames/001/12-104704_0/img_00032.jpg
1,2,frames/001/12-104704_1/img_00037.jpg
2,1,frames/001/12-104704_2/img_00038.jpg
3,1,frames/001/12-104704_3/img_00057.jpg
4,2,frames/001/12-104704_4/img_00046.jpg
...,...,...
759,2,frames/063/16-002607_0/img_00016.jpg
760,3,frames/063/16-002607_1/img_00063.jpg
761,4,frames/063/16-002607_2/img_00054.jpg
762,2,frames/063/16-002607_3/img_00048.jpg


In [78]:
#Scale UPDRS
df["UPDRS_score"] = df["UPDRS_score"] / 4

In [79]:
df.to_csv("df.csv", index = False)

In [80]:
#Checking labels
print(df["UPDRS_score"].value_counts())


UPDRS_score
0.25    891
0.50    766
0.00    547
0.75    512
1.00     95
Name: count, dtype: int64


In [81]:
#Defining labels and features 
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df, test_size = 0.2, random_state = 42)


In [82]:
#Building a custom generator to preprocess and generate image 

from tensorflow.keras.utils import Sequence
from tensorflow.keras.preprocessing.image import load_img, img_to_array
import os
#Defining the image class

class ImageGenerator(Sequence):
    #defining constructor
    def __init__(self, dataframe, batch_size = 32, img_size = (224, 224), shuffle = True, base_dir = ""):
        #Attrbutes from the parameters
        self.df = dataframe.reset_index(drop = True)
        self.batch_size = batch_size
        self.img_size = img_size
        self.shuffle = shuffle
        self.indices = np.arange(len(self.df))
        self.base_dir = base_dir
    
    #Defining methods 

    #defining the length by taking the samples and dividing by the batch_size 
    def __len__(self):
        return int(np.ceil(len(self.df) / self.batch_size)) - 50
    
    #retrieve each df by the indices
    def __getitem__(self, index):
        batch_indices = self.indices[index*self.batch_size:(index+1)*self.batch_size]
        batch_df = self.df.iloc[batch_indices]
    
        images = []
        labels = []
   #for each tuple in the dataframe,
        for _, row in batch_df.iterrows():
        #access the image from df
            img_path = os.path.join(self.base_dir, row['main_frame'])
            try:
                #load image 
                img = load_img(img_path, color_mode='grayscale', target_size=self.img_size)
                 # Normalize to [0,1]
                image = img_to_array(img) / 255.0  
                #append the image to the list
                images.append(image)
                #append the corresponding score
                labels.append(row['UPDRS_score'])
            except Exception as e:
                #print any error message
                print("Error loading", img_path)

        #Convert the list into an array for modeling
        print(np.array(images).shape, np.array(labels).shape)
        return np.array(images), np.array(labels, dtype = np.float32)
        
    #for each epoch, if shuffle is true
    def on_epoch_end(self):
        if self.shuffle:
            #randomly shuffle through the indices
            np.random.shuffle(self.indices)

In [83]:
#Balancing labels
labels = df['UPDRS_score'].values

from sklearn.utils import class_weight

class_weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(labels),
    y=labels
)
class_weights = dict(enumerate(class_weights))


In [84]:
#Initialize image gnerator with train datasets
directory = "/users/imbahndu/Desktop/Columbia DBM/PFED5"
train_gen = ImageGenerator(train_df, batch_size=32, img_size=(224, 224), base_dir = directory)
test_gen = ImageGenerator(test_df, batch_size=32, img_size=(224, 224))


In [85]:
#Building up the layers

cnn_model = keras.Sequential()

#Building 1st layer
input_layer = keras.layers.InputLayer(input_shape = (224, 224, 1))
cnn_model.add(input_layer)

#Building 2nd layer
hidden_layer_1 = keras.layers.Conv2D(filters = 16, kernel_size =3 , padding = "same")
batchNorm_1 = keras.layers.BatchNormalization()
relu_1 = keras.layers.ReLU()
cnn_model.add(hidden_layer_1)
cnn_model.add(batchNorm_1)
cnn_model.add(relu_1)

#Pooling layer 
cnn_model.add(keras.layers.MaxPooling2D(pool_size=(2, 2)))

#Building 3rd layer
hidden_layer_2 = keras.layers.Conv2D(filters = 32, kernel_size = 3, padding = "same")
batchNorm_2 = keras.layers.BatchNormalization()
relu_2 = keras.layers.ReLU()
cnn_model.add(hidden_layer_2)
cnn_model.add(batchNorm_2)
cnn_model.add(relu_2)

#Building 4th layer
hidden_layer_3 = keras.layers.Conv2D(filters = 64, kernel_size = 3, padding = "same")
batchNorm_3 = keras.layers.BatchNormalization()
relu_3 = keras.layers.ReLU()
cnn_model.add(hidden_layer_3)
cnn_model.add(batchNorm_3)
cnn_model.add(relu_3)

#Pooling layer
cnn_model.add(keras.layers.MaxPooling2D(pool_size=(2, 2)))

#Building 5th layer
hidden_layer_4 = keras.layers.Conv2D(filters = 128, kernel_size = 3, padding = "same")
batchNorm_4 = keras.layers.BatchNormalization()
relu_4 = keras.layers.ReLU()
cnn_model.add(hidden_layer_4)
cnn_model.add(batchNorm_4)
cnn_model.add(relu_4)

#Adding pooling layer for translation invariance
pooling_layer = keras.layers.GlobalAveragePooling2D()
cnn_model.add(pooling_layer)

#Regularization
cnn_model.add(keras.layers.Dropout(0.5))

#Adding output layer
output_layer = keras.layers.Dense(1, activation="linear")
cnn_model.add(output_layer)


#Summary 
cnn_model.summary()

/users/imbahndu/.local/lib/python3.9/site-packages/keras/src/layers/core/input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_20 (Conv2D)              │ (None, 224, 224, 16)   │           160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_20          │ (None, 224, 224, 16)   │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_20 (ReLU)                 │ (None, 224, 224, 16)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_10 (MaxPooling2D) │ (None, 112, 112, 16)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_21 (Conv2D)              │ (None, 112, 112, 32)   │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_21          │ (None, 112, 112, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_21 (ReLU)                 │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_22 (Conv2D)              │ (None, 112, 112, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_22          │ (None, 112, 112, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_22 (ReLU)                 │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_11 (MaxPooling2D) │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_23 (Conv2D)              │ (None, 56, 56, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_23          │ (None, 56, 56, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_23 (ReLU)                 │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_5      │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 98,241 (383.75 KB)

 Trainable params: 97,761 (381.88 KB)

 Non-trainable params: 480 (1.88 KB)

In [86]:
#Defining loss and optimizer

loss_fn = keras.losses.SparseCategoricalCrossentropy(from_logits = True)

optim = keras.optimizers.SGD(learning_rate = 0.0001)

In [87]:
#Compiling the model
cnn_model.compile(optimizer = "adam", loss = "mae" , metrics = ["mae"])

In [88]:
#Fitting the model 
epochs = 3
cnn_model.fit(train_gen, epochs = epochs, class_weight = class_weights)

(32, 224, 224, 1) (32,)
(32, 224, 224, 1) (32,)
Epoch 1/3


/users/imbahndu/.local/lib/python3.9/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


(32, 224, 224, 1) (32,)
(32, 224, 224, 1) (32,)
 1/21 ━━━━━━━━━━━━━━━━━━━━ 2:42 8s/step - loss: 0.9684 - mae: 1.0072(32, 224, 224, 1) (32,)
 2/21 ━━━━━━━━━━━━━━━━━━━━ 1:13 4s/step - loss: 0.8844 - mae: 0.9198(32, 224, 224, 1) (32,)
 3/21 ━━━━━━━━━━━━━━━━━━━━ 1:10 4s/step - loss: 0.8256 - mae: 0.8637(32, 224, 224, 1) (32,)
 4/21 ━━━━━━━━━━━━━━━━━━━━ 1:06 4s/step - loss: 0.7881 - mae: 0.8275(32, 224, 224, 1) (32,)
 5/21 ━━━━━━━━━━━━━━━━━━━━ 1:03 4s/step - loss: 0.7606 - mae: 0.8006(32, 224, 224, 1) (32,)
 6/21 ━━━━━━━━━━━━━━━━━━━━ 59s 4s/step - loss: 0.7391 - mae: 0.7801 (32, 224, 224, 1) (32,)
 7/21 ━━━━━━━━━━━━━━━━━━━━ 55s 4s/step - loss: 0.7216 - mae: 0.7632(32, 224, 224, 1) (32,)
 8/21 ━━━━━━━━━━━━━━━━━━━━ 51s 4s/step - loss: 0.7058 - mae: 0.7475(32, 224, 224, 1) (32,)
 9/21 ━━━━━━━━━━━━━━━━━━━━ 47s 4s/step - loss: 0.6924 - mae: 0.7347(32, 224, 224, 1) (32,)
10/21 ━━━━━━━━━━━━━━━━━━━━ 43s 4s/step - loss: 0.6806 - mae: 0.7233(32, 224, 224, 1) (32,)
11/21 ━━━━━━━━━━━━━━━━━━━━ 39s 4s/st